<a href="https://colab.research.google.com/github/arjunmenonv/Triton-Tutorial-ACM-Winter25/blob/main/HelloTriton.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# From Python to PTX: Dissecting Triton GPU Kernels with Compiler Explorer

ACM India Winter School on Compiler Design

Pradeep Ramachandran, Arjun Vadakkeveedu ---> me

KLA Advanced Computing Labs (AI-ACL), Chennai

December 22, 2025

## Contact Me!

I'm happy to connect with you, discuss and geek out on papers and ideas in GPU programming, compilers and computer architecture. You can reach out to me via:
- [LinkedIn](https://in.linkedin.com/in/arjun-menon-vadakkeveedu)
- Email: arjunmenonv@alumni.iitm.ac.in

What we are going to cover today:


1. GPU Hardware and Programming Model
2. Tile-Based Programming with Triton
3. Writing your first GPU kernel (vector add)
4. How your triton kernel gets mapped to hardware (godbolt)
5. Triton kernel to transpose a matrix
6. Matrix Multiplication in Triton



## Building a Mental Model for GPUs: Some useful resources

Hardware Abstraction:
- [CUDA C Programming Guide: Hardware Model](https://docs.nvidia.com/cuda/cuda-programming-guide/01-introduction/programming-model.html#gpu-hardware-model)
- [The Hopper H100 GPU Architecture in-depth](https://developer.nvidia.com/blog/nvidia-hopper-architecture-in-depth/#h100_sm_architecture)
- [Computer Architecture Lecture on GPU Architectures at ETH Zurich](https://www.youtube.com/live/gy6brWRgrnM?si=FyfqBbcpfjb-z4td&t=3206)

Software Abstraction:
- [Intro to CUDA C++](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/intro-to-cuda-cpp.html#intro-to-cuda-c)
- [OpenAI Triton](https://triton-lang.org/main/index.html)
- [GPU Programming @ IIT Madras by Prof. Rupesh Nasre](https://www.cse.iitm.ac.in/~rupesh/teaching/gpu/jan25/)

## Things to keep in mind when writing CUDA

- Data movement: Memory coalescing, Bank Conflicts
- Race conditions and synchronization
- Parallelism vs Occupancy trade-off
- Newer, more complicated execution and instruction modes (asynchronous, warp specialization, Tensor Core programming, ...)

## Environment Setup

In the `Runtime` tab, click on `Change runtime type` and select the following configuration:
- Runtime Type: Python3
- Hardware Accelerator: T4 GPU
- Runtime version: Latest

In [ ]:
!pip install triton==3.1.0
import torch
import triton

if torch.cuda.is_available():
    print("GPU runtime is enabled!")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU runtime is NOT enabled. Please ensure you have selected a GPU runtime type.")

print(f"Triton version: {triton.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.6/209.6 MB 5.0 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.5.0
    Uninstalling triton-3.5.0:
      Successfully uninstalled triton-3.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.9.0+cu126 requires triton==3.5.0; platform_system == "Linux", but you have triton 3.1.0 which is incompatible.
GPU runtime is enabled!
Device name: Tesla T4
Triton version: 3.1.0


In [ ]:
!nvidia-smi

Mon Dec 22 10:42:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Vector Addition in OpenAI Triton

Given two vectors `A` and `B` of length `N`, compute `C = A + B`.

`N` = 1024*1024 (1 Million)

Work out a mapping to split the input and output data into blocks, and complete the triton kernel skeleton below:

In [ ]:
import triton
import triton.language as tl

@triton.jit
def vec_add_kernel(x_ptr, y_ptr, out_ptr, n_elements: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    pass

In [ ]:
import torch

def vec_add(x: torch.Tensor, y: torch.Tensor, block_size: int = 256) -> torch.Tensor:
    assert x.is_cuda and y.is_cuda, "Input tensors must be CUDA tensors"
    assert x.shape == y.shape, "x and y must have the same shape"
    assert x.dtype == y.dtype, "x and y must have the same dtype"

    n = x.numel()
    out = torch.empty_like(x)

    # Launch the triton kernel

    return out


In [ ]:
numel = 1024*1024
BLOCK_SIZE = 1024
x = torch.rand(numel, device='cuda', dtype=torch.float32)
y = torch.rand(numel, device='cuda', dtype=torch.float32)
torch_out = x + y
triton_out = vec_add(x, y, block_size=BLOCK_SIZE)
err = torch.max(torch.abs(triton_out - torch_out))
print(f"Max error: {err.item()}")

Max error: 0.0


## Hello Godbolt!

Probably the most important thing you should take away as a compiler / performance geek from this session: [Compiler Explorer](https://godbolt.org/)

[Link to compiler-explorer window](https://godbolt.org/z/oTd6jvsab)

- Triton Compilation Flow through IRs
- Opt Pipeline
- Constexpr
- Reading PTX code

## Transposing a Matrix in Triton

Input: Matrix `A` of shape M × N

Output: Transposed A: shape N × M

In [ ]:
@triton.jit
def transpose_kernel(image: tl.tensor,
                    out: tl.tensor,
                    H: tl.int32,
                    W: tl.int32,
                    BLOCK_SIZE_H: tl.constexpr,
                    BLOCK_SIZE_W: tl.constexpr,
                    ):
    '''
        Input: NCHW image
        Output: NCWH image
        Expected Launch Config:
            Block: (BLOCK_SIZE_W, BLOCK_SIZE_H, 1) => (16, 16, 1)
            Grid: (ceil(W / BLOCK_SIZE_W), ceil(H / BLOCK_SIZE_H), N*C)
    '''
    pass

## Godbolt

[Link to compiler-explorer window](https://godbolt.org/z/xqhKG4f5z)

- Memory Coalescing
- Shared Memory

## General Matrix-Matrix Multiplication on GPU Tensor Cores

Fast GPU algorithms for Matrix Multiplications have been a key driving force behind the modern AI boom, and have contributed towards Nobel Prizes in Physics and Chemistry :).

Given matrices $\mathbf{A} \in \mathbb{R}^{M\times K}$ and $\mathbf{B} \in \mathbb{R}^{K\times N}$, compute their matrix product $\mathbf{C} = \mathbf{A} \mathbb{\times} \mathbf{B}$, $\mathbf{C} \in \mathbb{R}^{M\times N}$.

- M ∈ [32, 2048]
- N ∈ [32, 2048]
- K ∈ [32, 512]

The matrices $\mathbf{A}, \mathbf{B}$ and $\mathbf{C}$ can be in IEEE 32-bit or 16-bit floating point formats (`torch.float32` and `torch.float16` respectively).

Use a tiled implementation to improve the cache locality of the kernel. Pick the sizes of the data-tile (number of elements operated on by a block) and threadblock (number of threads used to service each data tile) appropriately to optimize the kernel's runtime.

In [ ]:
@triton.jit
def matmul_kernel(inpA: tl.tensor,
                inpB: tl.tensor,
                out: tl.tensor,
                M: tl.int32,
                N: tl.int32,
                K: tl.int32,
                BLOCK_M: tl.constexpr,
                BLOCK_N: tl.constexpr,
                BLOCK_K: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    #
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in tl.range(0, K, step=BLOCK_K):
        offs_k = k + tl.arange(0, BLOCK_K)
        #
        tileA = tl.load(inpA + offs_m[:, None] * K + offs_k[None, :],
                    mask= (offs_m[:, None] < M) & (offs_k[None, :] < K), other= 0.0)
        tileB = tl.load(inpB + offs_k[:, None] * N + offs_n[None, :],
                    mask= (offs_k[:, None] < K) & (offs_n[None, :] < N), other= 0.0)
        acc += tl.dot(tileA, tileB)
    tl.store(out + offs_m[:, None] * N + offs_n[None, :], acc,
                    mask= (offs_m[:, None] < M) & (offs_n[None, :] < N))

## Godbolt

[Link to compiler-explorer window](https://godbolt.org/z/dPMfe5ob4)

- Tiling
- Shared Memory Layouts